In [2]:
import pandas as pd
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
    "IMDB Dataset.csv"
)

# i need to clean the reviews first
# remove html tags and make everything lowercase
def clean_review(text):
    text = text.lower()
    # remove html tags
    text = re.sub(r"<.*?>", " ", text)
    # remove anything that isnt a letter
    text = re.sub(r"[^a-z]+", " ", text)
    words = text.split()
    return set(words)  # using a set so we dont count duplicates

# keywords we chose - positive ones and negative ones
positive_keywords = ["excellent", "wonderful", "brilliant", "perfect"]
negative_keywords = ["awful", "boring", "worst", "waste"]
all_keywords = positive_keywords + negative_keywords

# count total reviews
total_reviews = len(df)
total_positive = 0
total_negative = 0

for sentiment in df["sentiment"]:
    if sentiment == "positive":
        total_positive += 1
    else:
        total_negative += 1

print(f"Total reviews: {total_reviews}")
print(f"Positive: {total_positive}, Negative: {total_negative}")

# for each keyword, count how many positive and negative reviews contain it
keyword_counts = {}
for word in all_keywords:
    keyword_counts[word] = {"positive": 0, "negative": 0}

for i in range(len(df)):
    review = df["review"][i]
    sentiment = df["sentiment"][i]
    words_in_review = clean_review(review)

    for word in all_keywords:
        if word in words_in_review:
            keyword_counts[word][sentiment] += 1

# now compute bayes theorem for each keyword
# P(Positive | keyword) = P(keyword | Positive) * P(Positive) / P(keyword)

prior = total_positive / total_reviews  # P(Positive)
print(f"\nPrior P(Positive) = {prior:.4f}")
print("-" * 60)

for word in all_keywords:
    pos_count = keyword_counts[word]["positive"]
    neg_count = keyword_counts[word]["negative"]
    total_with_word = pos_count + neg_count

    # likelihood: if review is positive, how often does this word show up
    likelihood = pos_count / total_positive

    # marginal: how common is this word overall
    marginal = total_with_word / total_reviews

    # posterior: updated probability after seeing the keyword
    if marginal == 0:
        posterior = 0
    else:
        posterior = (likelihood * prior) / marginal

    print(f"Keyword: '{word}'")
    print(f"  P(Positive)          = {prior:.4f}")
    print(f"  P(keyword|Positive)  = {likelihood:.4f}")
    print(f"  P(keyword)           = {marginal:.4f}")
    print(f"  P(Positive|keyword)  = {posterior:.4f}")
    print()

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Total reviews: 50000
Positive: 25000, Negative: 25000

Prior P(Positive) = 0.5000
------------------------------------------------------------
Keyword: 'excellent'
  P(Positive)          = 0.5000
  P(keyword|Positive)  = 0.1147
  P(keyword)           = 0.0710
  P(Positive|keyword)  = 0.8074

Keyword: 'wonderful'
  P(Positive)          = 0.5000
  P(keyword|Positive)  = 0.0903
  P(keyword)           = 0.0556
  P(Positive|keyword)  = 0.8122

Keyword: 'brilliant'
  P(Positive)          = 0.5000
  P(keyword|Positive)  = 0.0635
  P(keyword)           = 0.0418
  P(Positive|keyword)  = 0.7601

Keyword: 'perfect'
  P(Positive)          = 0.5000
  P(keyword|Positive)  = 0.0817
  P(keyword)           = 0.0534
  P(Positive|keyword)  = 0.7657

Keyword: 'awful'
  P(Positive)          = 0.5000
  P(keyword|Positive)  = 0.0114
  P(keyword)           = 0.0577
  P(Positive|keyword)  = 0.0985

Keyword: 'boring'
  P(Pos